# emission

> Graph-root emission (CR-18 revolution 2): a completed source EMITS `Source -> AudioSegment -> Transcript` into the shared context graph — the graph now BEGINS at transcription (where-graph-begins resolution: ingestion is the first EXTENDER that plants the root). Deterministic identity tuples make emission idempotent: re-runs (cache hits included) collide into verified no-ops instead of duplicating roots (the E13 hazard, relocated into graph creation and discharged).

In [ ]:
#| default_exp emission

In [ ]:
#| export
import logging
from typing import Any, Dict, List, Optional, Tuple

from cjm_plugin_system.core.queue import JobQueue
from cjm_context_graph_layer.grammar import spine_edges
from cjm_context_graph_layer.ops import extend_graph
from cjm_context_graph_layer.declare import Derivation, derivation_to_graph
from cjm_transcript_graph_schema.schema import (
    SourceNode, AudioSegmentNode, AudioRenditionNode, TranscriptNode,
)

from cjm_transcription_core.models import SourceResult

logger = logging.getLogger(__name__)

In [ ]:
#| export
def build_source_emission(
    src: SourceResult,                          # Completed per-source pipeline result (0.3.0 shape)
    transcriber_config_hashes: Dict[str, str],  # transcriber -> effective config hash (Transcript identity input)
    chain: Optional[List[str]] = None,          # Preprocessing chain that produced the model-inputs ([]/None = raw convert-only)
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], Dict[str, Any]]:  # (nodes, edges, ids)
    """Build the graph-root payload for one source (pure; no capability calls).

    Emits the locked layer schema: one `Source` (identity = file content hash)
    -> coarse `AudioSegment` boundary spine (PART_OF / NEXT / STARTS_WITH via
    `spine_edges`) -> one `AudioRendition` per segment (the model-input WAV; it
    DERIVED_FROM its AudioSegment) -> per-transcriber `Transcript` variants
    (DERIVED_FROM their rendition; identity mirrors the capability cache key).
    Returns the ids dict {"source", "audio_segments", "renditions",
    "transcripts"} for callers (decomp recomputes these same ids from the
    manifest — no stored-id coupling).

    `chain` is the AudioRendition IDENTITY input: an empty chain is the raw
    convert-only rendition; a non-empty chain (e.g. demucs vocals) yields a
    DISTINCT rendition node under the SAME AudioSegment, so raw + preprocessed
    model-inputs of one boundary COEXIST in one graph (the divergent
    model_input_hash now lives on distinct rendition nodes, not a colliding
    AudioSegment). The chain rides into the node id, so re-derivation reproduces
    it from the manifest alone.
    """
    if not src.content_hash:
        raise ValueError(f"source {src.source_path} has no content_hash — emission identity requires it")
    chain = list(chain or [])
    source = SourceNode(content_hash=src.content_hash, path=src.source_path)
    nodes: List[Dict[str, Any]] = [source.to_graph_node()]
    edges: List[Dict[str, Any]] = []
    aseg_ids: List[str] = []
    rendition_ids: List[str] = []
    transcript_ids: Dict[str, List[str]] = {}

    for rec in src.segments:
        if not rec.model_input_hash:
            raise ValueError(f"segment {rec.index} has no model_input_hash — emission identity requires it")
        aseg = AudioSegmentNode(
            source=source.id, index=rec.index, start=rec.start, end=rec.end,
            segment_path=rec.segment_path,
        )
        nodes.append(aseg.to_graph_node())
        aseg_ids.append(aseg.id)
        # The model-input WAV is the rendition's, not the boundary's.
        rendition = AudioRenditionNode(
            audio_segment=aseg.id, model_input_path=rec.model_input_path,
            model_input_hash=rec.model_input_hash, chain=chain,
        )
        nodes.append(rendition.to_graph_node())
        edges.append(rendition.derived_edge())  # rendition DERIVED_FROM its AudioSegment
        rendition_ids.append(rendition.id)
        for tname, tr in rec.transcripts.items():
            tnode = TranscriptNode(
                rendition=rendition.id, transcriber=tname,
                config_hash=transcriber_config_hashes.get(tname, ""),
                text=str(tr.get("text") or ""), audio_hash=rec.model_input_hash,
                metadata=dict(tr.get("metadata") or {}),
            )
            nodes.append(tnode.to_graph_node())
            edges.append(tnode.derived_edge())  # transcript DERIVED_FROM its rendition
            transcript_ids.setdefault(tname, []).append(tnode.id)

    edges = spine_edges(source.id, aseg_ids) + edges
    ids = {"source": source.id, "audio_segments": aseg_ids,
           "renditions": rendition_ids, "transcripts": transcript_ids}
    return nodes, edges, ids

In [ ]:
#| export
async def emit_source_graph(
    queue: JobQueue,                            # Started job queue
    graph_id: str,                              # Graph-storage capability instance id
    src: SourceResult,                          # Completed per-source pipeline result
    transcriber_config_hashes: Dict[str, str],  # transcriber -> effective config hash
    run_id: str,                                # Run id (recorded on the boundary Derivation event)
    chain: Optional[List[str]] = None,          # Preprocessing chain that produced the model-inputs ([]/None = raw)
) -> Dict[str, Any]:  # Emission record for the manifest
    """Idempotently emit one source's graph root through the task channel.

    `extend_graph` = emit-if-absent + verify-if-present, so a re-run over
    cached content collides into a verified no-op (stress item 4) and a second
    transcriber's run EXTENDS the existing root (only its Transcript nodes are
    new). The host's contribution this run — boundary computation and/or the
    preprocessing chain that produced new renditions — is declared as a
    `Derivation` event (provenance-by-declaration) ONLY when it actually created
    AudioSegment or AudioRendition nodes (a preprocessing-ON run into a graph
    that already holds the raw boundaries creates new renditions but no new
    boundaries — both cases declare; verified re-emissions don't spam the audit
    trail).
    """
    chain = list(chain or [])
    nodes, edges, ids = build_source_emission(src, transcriber_config_hashes, chain=chain)
    res = await extend_graph(queue, graph_id, nodes, edges)
    added = set(res.added_node_ids)
    new_asegs = [a for a in ids["audio_segments"] if a in added]
    new_renditions = [r for r in ids["renditions"] if r in added]
    if new_asegs or new_renditions:
        parts = (["segment-boundaries"] if new_asegs else []) + (["preprocessing"] if (chain and new_renditions) else [])
        method = ("+".join(parts) or "renditions") + "/v1"
        props: Dict[str, Any] = {"run_id": run_id}
        if chain:
            props["chain"] = list(chain)
        d = Derivation(
            actor="host:cjm-transcription-core", method=method,
            input_ids=[ids["source"]], output_ids=new_asegs + new_renditions,
            properties=props,
        )
        dn, de = derivation_to_graph(d)
        await extend_graph(queue, graph_id, [dn], de)
    record = {
        "source_node_id": ids["source"],
        "nodes_added": res.nodes_added,
        "nodes_verified": res.nodes_verified,
        "edges_added": res.edges_added,
        "edges_existing": res.edges_existing,
    }
    logger.info(f"emitted {src.source_path}: {record}")
    return record

In [ ]:
# tests — emission payload shape + identity determinism (pure; no plugins)
from cjm_transcription_core.models import SegmentRecord
from cjm_transcript_graph_schema.schema import (
    source_node_id, audio_segment_node_id, audio_rendition_node_id, transcript_node_id,
)

_recs = [
    SegmentRecord(index=0, start=0.0, end=280.0, duration=280.0,
                  segment_path="/cuts/s0.mp3", model_input_path="/cache/s0.wav",
                  model_input_hash="sha256:wav0",
                  transcripts={"whisper": {"job_id": "j0w", "text": "hello", "metadata": {}},
                               "voxtral": {"job_id": "j0v", "text": "hullo", "metadata": {}}}),
    SegmentRecord(index=1, start=280.0, end=560.0, duration=280.0,
                  segment_path="/cuts/s1.mp3", model_input_path="/cache/s1.wav",
                  model_input_hash="sha256:wav1",
                  transcripts={"whisper": {"job_id": "j1w", "text": "world", "metadata": {}},
                               "voxtral": {"job_id": "j1v", "text": "wurld", "metadata": {}}}),
]
_src = SourceResult(source_path="/media/ep1.mp3", duration=560.0, vad_chunk_count=99,
                    batch_key="bk", content_hash="sha256:src", segments=_recs)
_hashes = {"whisper": "sha256:cfgw", "voxtral": "sha256:cfgv"}

nodes, edges, ids = build_source_emission(_src, _hashes)
# 1 Source + 2 AudioSegment + 2 AudioRendition + 2x2 Transcript
assert len(nodes) == 9
labels = [n["label"] for n in nodes]
assert labels.count("Source") == 1 and labels.count("AudioSegment") == 2
assert labels.count("AudioRendition") == 2 and labels.count("Transcript") == 4
# spine: 1 STARTS_WITH + 2 PART_OF + 1 NEXT; plus 2 rendition + 4 transcript DERIVED_FROM = 6
rels = [e["relation_type"] for e in edges]
assert rels.count("STARTS_WITH") == 1 and rels.count("PART_OF") == 2
assert rels.count("NEXT") == 1 and rels.count("DERIVED_FROM") == 6
# AudioSegment is a hashless boundary (model-input moved to the rendition); raw chain -> is_raw rendition
assert all(n["sources"] == [] and "model_input_path" not in n["properties"]
           for n in nodes if n["label"] == "AudioSegment")
assert all(n["properties"]["is_raw"] is True and n["sources"][0]["content_hash"].startswith("sha256:")
           for n in nodes if n["label"] == "AudioRendition")

# deterministic ids recomputable from the manifest fields alone
assert ids["source"] == source_node_id("sha256:src")
a0 = audio_segment_node_id(ids["source"], 0.0, 280.0)
assert ids["audio_segments"][0] == a0
r0 = audio_rendition_node_id(a0, [])  # raw rendition
assert ids["renditions"][0] == r0
assert ids["transcripts"]["whisper"][0] == transcript_node_id(r0, "whisper", "sha256:cfgw")
# re-build -> byte-identical id sets (emission idempotency precondition)
nodes2, edges2, ids2 = build_source_emission(_src, _hashes)
assert [n["id"] for n in nodes2] == [n["id"] for n in nodes]
assert [e["id"] for e in edges2] == [e["id"] for e in edges]

# preprocessing chain: DISTINCT renditions/transcripts under the SAME AudioSegments (coexist).
_chain = ["source_separation:cjm-media-plugin-demucs@cfg123"]
nodes_p, edges_p, ids_p = build_source_emission(_src, _hashes, chain=_chain)
assert ids_p["audio_segments"] == ids["audio_segments"]  # boundary shared across renditions
assert ids_p["renditions"] != ids["renditions"]          # vocals renditions are distinct nodes
assert ids_p["transcripts"]["whisper"] != ids["transcripts"]["whisper"]
assert ids_p["renditions"][0] == audio_rendition_node_id(a0, _chain)
assert all(n["properties"]["is_raw"] is False and n["properties"]["preprocessing"] == _chain[0]
           for n in nodes_p if n["label"] == "AudioRendition")
# raw + vocals payloads share zero rendition ids -> they can land in ONE graph without collision
assert not (set(ids["renditions"]) & set(ids_p["renditions"]))

# identity guards fire loudly
import dataclasses
try:
    build_source_emission(dataclasses.replace(_src, content_hash=""), _hashes)
    raise AssertionError("expected ValueError")
except ValueError:
    pass
print("emission shape tests OK")